In [ ]:
import os
import base64
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from time import sleep, time as now

# === Load GitHub Tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found.")

token_index = 0
def get_headers():
    return {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "manifest-checker"
    }

def rotate_token():
    global token_index
    token_index = (token_index + 1) % len(tokens)
    print(f"🔁 Rotated to token #{token_index + 1}")

# === Make GitHub API Request with Retry and Rotation ===
def make_request(url):
    tries_flag = 0
    while True:
        res = requests.get(url, headers=get_headers())
        if res.status_code == 200:
            return res
        elif res.status_code == 403 and "rate limit" in res.json().get("message", "").lower():
            rotate_token()
            tries_flag += 1
            if tries_flag > len(tokens):
                reset_timestamp = int(res.headers.get("X-RateLimit-Reset", now() + 60))
                wait_time = reset_timestamp - int(now())
                print(f"⏳ Rate limit hit. Sleeping for {wait_time} seconds...")
                sleep(wait_time + 1)
                tries_flag = 0
            else:
                sleep(1)
        elif res.status_code in [404, 422]:
            return None
        else:
            print(f"⚠️ Request failed: {res.status_code}. Retrying...")
            sleep(1)

# === Check for manifest and activity
def check_manifest_and_activity(full_name):
    search_url = f"https://api.github.com/search/code?q=filename:AndroidManifest.xml+repo:{full_name}"
    search_res = make_request(search_url)
    if not search_res:
        return "no", "no", "no", 0

    data = search_res.json()
    manifest_count = data.get("total_count", 0)
    if manifest_count == 0:
        return "no", "no", "no", 0

    found_activity = False
    found_standard = False

    for item in data.get("items", []):
        path = item.get("path", "").lower()
        if path in ["app/src/main/androidmanifest.xml", "src/main/androidmanifest.xml"]:
            found_standard = True
        file_url = item.get("url")
        file_res = make_request(file_url)
        if not file_res:
            continue
        content = file_res.json().get("content")
        if not content:
            continue
        try:
            xml_text = base64.b64decode(content).decode("utf-8", errors="ignore")
            root = ET.fromstring(xml_text)
            if root.findall(".//activity"):
                found_activity = True
                break
        except ET.ParseError:
            continue

    return (
        "yes",
        "yes" if found_activity else "no",
        "yes" if found_standard else "no",
        manifest_count
    )

# === File paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step4_keyword_check_output.csv"
interim_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step5_manifest_check_output_interim.csv"
final_output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step5_manifest_check_output.csv"

# === Load Data ===
if os.path.exists(interim_path):
    print(f"🔄 Resuming from interim: {interim_path}")
    df = pd.read_csv(interim_path)
else:
    df = pd.read_csv(input_path)
    df["has_manifest"] = "none"
    df["has_activity"] = "none"
    df["standard_manifest"] = "none"
    df["manifest_count"] = -1
    df["Valid_Repo_Step5"] = "none"

# === Filter: only review repos with Valid_Repo_Step4 == "yes"
valid_df = df[(df["Valid_Repo_Step4"] == "yes") & (df["manifest_count"] == -1)].copy()
print(f"🔍 Reviewing {len(valid_df)} repos with Valid_Repo_Step4 == 'yes'\n")

# === Process each repo ===
for i, row in enumerate(valid_df.itertuples(), start=1):
    idx = row.Index
    full_name = row.full_name
    print(f"\n🔎 [{i}/{len(valid_df)}] Checking: {full_name}")

    has_manifest, has_activity, standard_manifest, count = check_manifest_and_activity(full_name)

    df.at[idx, "has_manifest"] = has_manifest
    df.at[idx, "has_activity"] = has_activity
    df.at[idx, "standard_manifest"] = standard_manifest
    df.at[idx, "manifest_count"] = count

    if has_manifest == "yes" and has_activity == "yes":
        df.at[idx, "Valid_Repo_Step5"] = "yes"
    else:
        df.at[idx, "Valid_Repo_Step5"] = "no"

    if (i % 10 == 0) or (i =


🔄 Resuming from interim file: C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step5_manifest_check_output_interim.csv

🔍 [1/4300] Checking: googleapis/java-bigtable

🔍 [2/4300] Checking: googleapis/java-storage-nio

🔍 [3/4300] Checking: googleapis/java-bigquerystorage

🔍 [4/4300] Checking: f-droid/privileged-extension
📁 Found standard manifest path: app/src/main/androidmanifest.xml
❌ No <activity> in app/src/main/androidmanifest.xml
📄 Found non-standard manifest path: app/src/androidtest/androidmanifest.xml
❌ No <activity> in app/src/androidtest/androidmanifest.xml

🔍 [5/4300] Checking: ChandlerZeng/MyStock
📁 Found standard manifest path: app/src/main/androidmanifest.xml
✅ Found <activity> in app/src/main/androidmanifest.xml

🔍 [6/4300] Checking: SupernautApp/SupernautFX

🔍 [7/4300] Checking: javaexception/QzsWanAndroid
📁 Found standard manifest path: app/src/main/androidmanifest.xml
✅ Found <activity> in app/src/main/androidmanifest.xml

🔍 [8/4300] Checking: david4599